In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np


In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_data = datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_data  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=64, shuffle=False)

print(f"Train samples: {len(train_data)}")
print(f"Test samples:  {len(test_data)}")

100.0%
100.0%
100.0%
100.0%

Train samples: 60000
Test samples:  10000


In [5]:
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.view(-1, 784)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

model = MLP()
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

MLP(
  (fc1): Linear(in_features=784, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=128, bias=True)
  (fc3): Linear(in_features=128, out_features=10, bias=True)
  (relu): ReLU()
)

Total parameters: 235,146


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train(model, loader, optimizer, criterion, epochs=10):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        correct = 0
        for images, labels in loader:
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()

        acc = 100 * correct / len(loader.dataset)
        print(f"Epoch {epoch+1:2d}/10 | Loss: {total_loss/len(loader):.4f} | Accuracy: {acc:.2f}%")


train(model, train_loader, optimizer, criterion)

# save model weights after training
torch.save(model.state_dict(), '~/projects/digit-recognition-fpga/weights/model.pt')
print("Model saved to weights/model.pt")

Epoch  1/10 | Loss: 0.2289 | Accuracy: 93.06%
Epoch  2/10 | Loss: 0.0923 | Accuracy: 97.18%
Epoch  3/10 | Loss: 0.0645 | Accuracy: 97.98%
Epoch  4/10 | Loss: 0.0476 | Accuracy: 98.43%
Epoch  5/10 | Loss: 0.0407 | Accuracy: 98.68%
Epoch  6/10 | Loss: 0.0346 | Accuracy: 98.87%
Epoch  7/10 | Loss: 0.0286 | Accuracy: 99.06%
Epoch  8/10 | Loss: 0.0260 | Accuracy: 99.14%
Epoch  9/10 | Loss: 0.0212 | Accuracy: 99.27%
Epoch 10/10 | Loss: 0.0206 | Accuracy: 99.32%


RuntimeError: Parent directory weights does not exist.

In [ ]:
def test(model, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in loader:
            outputs = model(images)
            correct += (outputs.argmax(1) == labels).sum().item()
    print(f"Test Accuracy: {100 * correct / len(loader.dataset):.2f}%")

In [ ]:
def quantize_weights(tensor, num_bits=8):
    w_min = tensor.min().item()
    w_max = tensor.max().item()

    scale = (w_max - w_min) / (2**num_bits - 1)


    w_int = torch.round((tensor - w_min) / scale).to(torch.int32)

    return w_int.numpy(), scale, w_min

w1, s1, z1 = quantize_weights(model.fc1.weight.data)
w2, s2, z2 = quantize_weights(model.fc2.weight.data)
w3, s3, z3 = quantize_weights(model.fc3.weight.data)

print(f"Layer 1 weights shape: {w1.shape} | scale: {s1:.6f}")
print(f"Layer 2 weights shape: {w2.shape} | scale: {s2:.6f}")
print(f"Layer 3 weights shape: {w3.shape} | scale: {s3:.6f}")

Layer 1 weights shape: (256, 784) | scale: 0.000280
Layer 2 weights shape: (128, 256) | scale: 0.000490
Layer 3 weights shape: (10, 128) | scale: 0.000691


In [ ]:
np.save('w1_int8.npy', w1.astype(np.int8))
np.save('w2_int8.npy', w2.astype(np.int8))
np.save('w3_int8.npy', w3.astype(np.int8))


scales = np.array([s1, z1, s2, z2, s3, z3])
np.save('scales.npy', scales)

print("Saved: w1_int8.npy, w2_int8.npy, w3_int8.npy, scales.npy")



Saved: w1_int8.npy, w2_int8.npy, w3_int8.npy, scales.npy


In [ ]:

from google.colab import files
files.download('w1_int8.npy')
files.download('w2_int8.npy')
files.download('w3_int8.npy')
files.download('scales.npy')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
files.download('model.pt')